In [ ]:
import pandas as pd
import os
import numpy as np
import pickle
import math
from collections import defaultdict
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [ ]:
#MMR failed to find it so this reduces the data to work better
def calculate_idf(tokenized_data):
    num_docs = len(tokenized_data)
    doc_freq = defaultdict(int)
    
    for tokens in tokenized_data:
        unique_tokens = set(tokens)
        for token in unique_tokens:
            doc_freq[token] += 1
            
    idf_dict = {}
    for token, freq in doc_freq.items():
        idf_dict[token] = math.log(num_docs / freq) + 1.0
    return idf_dict

In [ ]:
def user_vectors():
    csv_path = os.path.join("Task 1", "mongodb", "user_vec.parquet")
    
    df = pd.read_parquet(csv_path)
    items = df['item_string'].fillna('').tolist()
    
    tokenized_data = [simple_preprocess(item) for item in items]
    
    model = Word2Vec(
        sentences=tokenized_data, 
        vector_size=100, 
        window=5, 
        min_count=1, 
        workers=4
    )
    
    model.save("word2vec_user.model")
    
    def get_average_vector(tokens, model):
        valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
        if len(valid_vectors) == 0:
            return np.zeros(model.vector_size)
        return np.mean(valid_vectors, axis=0)

    chunk_size = 5000
    all_vectors = []
    
    for i in range(0, len(tokenized_data), chunk_size):
        batch = tokenized_data[i : i + chunk_size]
        batch_vecs = [get_average_vector(tokens, model) for tokens in batch]
        all_vectors.extend(batch_vecs)
        print(f"Processed batch ending at row {i + len(batch)}")
    
    df_out = pd.DataFrame({
        'id': df['_id'].astype(str),
        'vector': [vec.tolist() for vec in all_vectors],
        'metadata': [{'user_id': str(uid)} for uid in df['_id']]
    })
    
    df_out.to_parquet("master_user_vectors_word2vec.parquet", engine="pyarrow")
    print("completed user vectors")

if __name__ == "__main__":
    user_vectors()

In [ ]:
def product_vectors():
    csv_path = os.path.join("..","..","Task 1", "mongodb", "product_vec.parquet")
    
    df = pd.read_parquet(csv_path)
    items = df['combined'].fillna('').tolist()
    
    tokenized_data = [simple_preprocess(item) for item in items]
    
    idf_dict = calculate_idf(tokenized_data)
    
    with open("word2vec_product_weights.pkl", "wb") as f:
        pickle.dump(idf_dict, f)
    

    model = Word2Vec(
        sentences=tokenized_data, 
        vector_size=100, 
        window=5, 
        min_count=2, 
        workers=4
    )
    model.save("word2vec_product.model")
    
    #apply weights
    def get_weighted_average_vector(tokens, model, idf_dict):
        valid_words = [word for word in tokens if word in model.wv and word in idf_dict]
        if not valid_words:
            return np.zeros(model.vector_size)
            
        vectors = [model.wv[word] * idf_dict[word] for word in valid_words]
        return np.mean(vectors, axis=0)
        
    chunk_size = 5000
    all_vectors = []
    
    for i in range(0, len(tokenized_data), chunk_size):
        batch = tokenized_data[i : i + chunk_size]
        batch_vecs = [get_weighted_average_vector(tokens, model, idf_dict) for tokens in batch]
        all_vectors.extend(batch_vecs)
        print(f"Processed product batch ending at row {i + len(batch)}")

    df_out = pd.DataFrame({
        'id': df['parent_asin'].astype(str),
        'vector': [vec.tolist() for vec in all_vectors],
        'combined': df['combined'],
        'title': df['title'].astype(str)
    })
    
    df_out.to_parquet("master_product_vectors_word2vec.parquet", engine="pyarrow")
    print("Completed product Word2Vec generation.")

if __name__ == "__main__":
    product_vectors()